<center><img src='https://drive.google.com/uc?id=1_utx_ZGclmCwNttSe40kYA6VHzNocdET' height="60"></center>

AI TECH - Akademia Innowacyjnych Zastosowań Technologii Cyfrowych. Program Operacyjny Polska Cyfrowa na lata 2014-2020
<hr>

<center><img src='https://drive.google.com/uc?id=1BXZ0u3562N_MqCLcekI-Ens77Kk4LpPm'></center>

<center>
Projekt współfinansowany ze środków Unii Europejskiej w ramach Europejskiego Funduszu Rozwoju Regionalnego
Program Operacyjny Polska Cyfrowa na lata 2014-2020,
Oś Priorytetowa nr 3 "Cyfrowe kompetencje społeczeństwa" Działanie  nr 3.2 "Innowacyjne rozwiązania na rzecz aktywizacji cyfrowej"
Tytuł projektu:  „Akademia Innowacyjnych Zastosowań Technologii Cyfrowych (AI Tech)”
    </center>

# MSLE and gradients

Today we're implementing Gradient Descent by hand.
We'll also look at the Mean Squared Logarithmic Error (MSLE).

In [1]:
%matplotlib inline

from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
from tqdm.auto import tqdm

NDArray = np.ndarray[Any, Any]

np.set_printoptions(precision=4, suppress=True)
np.random.seed(357)

## Loading and converting data

Same data as before, cleaned up. We'll use NumPy arrays instead of Pandas DataFrames to simplify the implementation.

In [2]:
!wget --no-verbose -O lab1_apartments_train_clean.csv https://mimuw.edu.pl/~mwrochna/upload/lab1_apartments_train_clean.csv
!wget --no-verbose -O lab1_apartments_test_clean.csv https://mimuw.edu.pl/~mwrochna/upload/lab1_apartments_test_clean.csv
!head lab1_apartments_train_clean.csv lab1_apartments_test_clean.csv

2026-10-14 14:38:51 URL:https://mimuw.edu.pl/~mwrochna/upload/lab1_apartments_train_clean.csv [6187/6187] -> "lab1_apartments_train_clean.csv" [1]
2026-10-14 14:38:52 URL:https://mimuw.edu.pl/~mwrochna/upload/lab1_apartments_test_clean.csv [6223/6223] -> "lab1_apartments_test_clean.csv" [1]
==> lab1_apartments_train_clean.csv <==
area,district,bedrooms,bathrooms,build_year,parking,price
104,mokotowo,2,2,1940,1,780094
43,ochotowo,1,1,1970,1,346912
128,grodziskowo,3,2,1916,1,523466
112,mokotowo,3,2,1920,1,830965
149,mokotowo,3,3,1977,0,1090479
80,ochotowo,2,2,1937,0,599060
58,ochotowo,2,1,1922,0,463639
23,ochotowo,1,1,1929,0,166785
40,mokotowo,1,1,1973,0,318849

==> lab1_apartments_test_clean.csv <==
area,district,bedrooms,bathrooms,build_year,parking,price
71,wolowo,2,2,1912,1,322227
45,mokotowo,1,1,1938,0,295878
38,mokotowo,1,1,1999,1,306530
70,ochotowo,2,2,1980,1,553641
136,mokotowo,3,2,1939,1,985348
128,wolowo,3,2,1983,1,695726
23,grodziskowo,1,1,1975,0,99751
117,mokotowo,3,2,1942,0,

In [3]:
def load_dataset(path: str) -> tuple[NDArray, NDArray]:
    """
    Returns (x, y) where:
    - x: input features, shape (n_apartments, n_features)
    - y: prices, shape (n_apartments,)
    """
    data = pd.read_csv(path)
    y = data["price"].to_numpy()
    x = data.loc[:, data.columns != "price"].to_numpy()
    return x, y

x_train, y_train = load_dataset("lab1_apartments_train_clean.csv")
x_test, y_test = load_dataset("lab1_apartments_test_clean.csv")

print(x_train.shape, y_train.shape)
print(x_test.shape, y_test.shape)

(200, 6) (200,)
(200, 6) (200,)


In [4]:
# Convert categorical features to numbers (one ordinal column, for simplicity).
ordinal_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

ordinal_encoder.fit(x_train[:, [1]])

x_train[:, [1]] = ordinal_encoder.transform(x_train[:, [1]])
x_test[:, [1]] = ordinal_encoder.transform(x_test[:, [1]])

# Convert ints to float.
x_train = x_train.astype(np.float64)
x_test = x_test.astype(np.float64)

# Preview the first three rows.
x_train[:3]

array([[ 104.,    1.,    2.,    2., 1940.,    1.],
       [  43.,    2.,    1.,    1., 1970.,    1.],
       [ 128.,    0.,    3.,    2., 1916.,    1.]])

## The loss, constant models

The range of prices is quite large.

In [5]:
print(np.min(y_train), np.max(y_train), np.mean(y_train))

102572 1102309 507919.49


When predicting them, relative errors are arguably more important than absolute errors.<br>
That is, mistaking 1M vs 2M is arguably similar to mistaking 11M vs 22M, more so than mistaking 11M vs 12M.

Therefore, let's assume that our predictions should minimize the so-called *mean squared logarithmic error*:
$$
MSLE(\bar{y}, \bar{p}) = \frac{1}{n} \sum_{i=1}^n (\log(1+y_i) - \log(1+p_i))^2,
$$
where $y_i$ is the ground truth (the target), and $p_i$ is our prediction.

Let's implement the loss function first.

In [6]:
def mse(ys: NDArray, ps: NDArray) -> np.float64:
    assert ys.shape == ps.shape
    return np.mean((ys - ps) * (ys - ps))

In [7]:
def msle(ys: NDArray, ps: NDArray) -> np.float64:
    assert ys.shape == ps.shape
    #################################################
    # TODO: Implement this function (without loops) #
    #################################################
    log_ys = np.log(1 + ys)
    log_ps = np.log(1 + ps)
    return np.mean((log_ys - log_ps) ** 2)


The simplest model is predicting the same constant for each instance. Test your implementation of msle against outputing the mean price.

In [8]:
###################################################
# TODO: Compute msle for outputing the mean price #
###################################################
msle(y_test, np.full_like(y_test, np.mean(y_train)))

np.float64(0.4284113768688732)

To get a feel of whether this error is high or low, we can apply exp and sqrt to get some kind of mean (`exp of quadratic mean of log`) in interpretable units.<br>
(For context, observe that `exp of arithmetic mean of log` is the geometric mean).

In [9]:
np.exp(np.sqrt(msle(y_test, np.full_like(y_test, np.mean(y_train)))))

np.float64(1.9242406411512714)

So we are roughly 1.92× off the actual y_test when we predict that constant value all the time, on average.

Recall that outputing the **mean** minimizes $MSE$. However, we're now dealing with $MSLE$.

Find a constant that should result in the lowest $MSLE$.

In [13]:
#############################################
# TODO: Find this constant and compute msle #
#############################################
# mean is a constant that minimizes the mse, but here we want to find constant that works with msle

costs = np.linspace(start=np.min(y_train), stop=np.max(y_train), num=100)
# for each we want to compute msle, and than we'll keep the minimum
losses = [msle(y_train, np.full_like(y_train, c)) for c in costs]

best_cost = costs[np.argmin(losses)]

print("test", msle(y_test, np.full_like(y_test, best_cost)))
print("train", msle(y_train, np.full_like(y_train, best_cost)))

test 0.4260799965189193
train 0.364991711231755


In [14]:
# to minimize mse(ys, c), take c = mean(ys)

# minimize msle(y_train, c) = mse(f(y_train), f(c)) over c where f(x) = log(1+x)
# log(1 + c) = f(c) = mean(f(y_train))
# c = f^{1}(mean(f(y_train))) = exp(mean(log(1 + y_train))) - 1

best_cost = np.exp(np.mean(np.log(y_train + 1))) - 1
print("test", msle(y_test, np.full_like(y_test, best_cost)))
print("train", msle(y_train, np.full_like(y_train, best_cost)))

test 0.4275753834952471
train 0.36488961221465716


## Linear regression (standard)

Now, let's implement training of a standard linear regression model via gradient descent.

In [20]:
# we need to initialize weights

def train(
    x: NDArray, y: NDArray, alpha: float = 1e-7, n_iterations: int = 100000
) -> tuple[NDArray, np.float64]:
    """Linear regression (which optimizes MSE). Returns (weights, bias)."""

    # B is batch size (number of observations).
    # D is number of (input) features.
    B, D = x.shape
    assert y.shape == (B,)

    ### TODO ###

    weights = np.random.uniform(low= (-1 / np.sqrt(D)), high= (1 / np.sqrt(D)), size= (D,))
    # why sqrt?
    # when we first compute the sum of w_1 + x_i + b --> the question is what scale should the waights have?
    # we usually need to stick to the scale between -1 and 1
    bias = np.mean(y) # or best_cost

    for i in range(n_iterations):
      # FORWARD PASS
      # shapes = (B, D) @ (D, ) + ()
      preds = x @ weights + bias
      # we want to have the sum
      # or pred_i = sum_j x_ij * w_j + bias

      loss = mse(preds, y) # 1 / B * sum(preds - y)**@

      # BACKWARD PASS
      # d loss / d {preds_i} =
      # (1 / B) * d ( ( preds_i - y_i) ** 2) / d pred_i
      # (2 / B) * (preds_i - y_i)
      grad_p = (2 / B) * (preds - y) # shape (B,)

      # d loss / d_{w_j} =

      #weights -= alpha * grad_w
      #bias -= alpha * grad_b

      grad_w = x.T @ grad_p

      grad_b = np.sum(grad_p)

      weights -= alpha * grad_w
      bias -= alpha * grad_b
    ### END ###
    return weights, bias

weights, bias = train(x_train, y_train)
preds_test = x_test @ weights + bias  # shapes (B, D) @ (D,) + ()
print("test MSLE:", msle(y_test, preds_test))
# Should be much better than the constant model.

test MSLE: 0.0787396140949444


## Linear regression (MSLE)

Note that the loss function that the algorithms optimizes (i.e $MSE$) differs from $MSLE$. We've already seen that this may result in a suboptimal solution.

How can you change the setting so that we optimze $MSLE$ instead?

Hint:
<sub><sup><sub><sup><sub><sup>
Be lazy. We don't want to change the algorithm.
Use the chain rule and previous computations to get formulas for the gradient.
</sup></sub></sup></sub></sup></sub>

In [ ]:
def train_msle(
    x: NDArray, y: NDArray, alpha: float = 1e+4, n_iterations: int = 50000
) -> tuple[NDArray, NDArray]:
    """Gradient descent for MSLE."""

    #############################################
    # TODO: Optimize msle and compare the error #
    #############################################

    # B is batch size (number of observations).
    # D is number of (input) features.
    B, D = x.shape
    assert y.shape == (B,)


    return weights, bias


weights, bias = train_msle(x_train, y_train)
preds_test = x_test @ weights + bias
print("test MSLE: ", msle(y_test, preds_test))
# Train MSLE should be better. Test MSLE not necessarily.